In [ ]:
!git clone https://github.com/NVlabs/GEM-X.git
%cd GEM-X

In [ ]:
# Init only sam-3d-body and soma — skip soma-retargeter
!git submodule update --init third_party/sam-3d-body
!git submodule update --init third_party/soma


In [ ]:
!ls third_party/sam-3d-body
!ls third_party/soma

In [ ]:
!apt-get install git-lfs -q
!git lfs install
%cd third_party/soma
!git lfs pull
%cd ../..

In [ ]:
!pip install -e third_party/soma

In [ ]:
!bash scripts/install_env.sh

In [ ]:
!pip install -e .

In [ ]:
!python -c "import gem; print('GEM-X installed ok')"
!python -c "from soma import SOMALayer; print('SOMA installed ok')"

In [ ]:
from google.colab import files
uploaded = files.upload()

# New Section

In [ ]:
!pip install onnxruntime

In [ ]:
%cd /content/GEM-X
!python scripts/demo/demo_soma.py --video /content/GEM-X/example2.mp4

CONVERTING SOMA TO MHR

In [ ]:
import os
# GEM-X saves outputs here
!find /content/GEM-X/outputs -name "soma_params.pt"
!ls /content/GEM-X/outputs/

In [ ]:
import torch

hpe = torch.load('/content/GEM-X/outputs/demo_soma/example2/hpe_results.pt', map_location='cpu')

print(type(hpe))
print(hpe.keys() if isinstance(hpe, dict) else dir(hpe))

# If dict, inspect all keys and shapes
if isinstance(hpe, dict):
    for k, v in hpe.items():
        if isinstance(v, dict):
            print(f"\n{k}:")
            for k2, v2 in v.items():
                print(f"  {k2}: {v2.shape if hasattr(v2, 'shape') else type(v2)}")
        elif hasattr(v, 'shape'):
            print(f"{k}: {v.shape}")
        else:
            print(f"{k}: {type(v)}")

In [ ]:
import torch

hpe = torch.load('/content/GEM-X/outputs/demo_soma/example2/hpe_results.pt', map_location='cpu')
body = hpe['body_params_global']

global_orient   = body['global_orient']    # (100, 3)
body_pose       = body['body_pose']        # (100, 228)
full_pose       = torch.cat([global_orient, body_pose], dim=-1)  # (100, 231)
identity_coeffs = body['identity_coeffs']  # (100, 45)
scale_params    = body['scale_params']     # (100, 69)
transl          = body['transl']           # (100, 3)

print(f"full_pose:       {full_pose.shape}")
print(f"identity_coeffs: {identity_coeffs.shape}")
print(f"scale_params:    {scale_params.shape}")
print(f"transl:          {transl.shape}")

WORKED CODE FOR CONVERSION


In [ ]:
!find /content/GEM-X -name "*.py" | xargs grep -l "SOMALayer" 2>/dev/null

EXAMPLE 1

In [ ]:
import sys

# Remove the wrong soma from cache
if 'soma' in sys.modules:
    del sys.modules['soma']

# Add GEM-X's third_party folder FIRST so it overrides the system soma
sys.path.insert(0, '/content/GEM-X/third_party/soma')


from soma import SOMALayer
import torch

soma_layer = SOMALayer(identity_model_type='mhr', device='cpu')

# Reshape pose from (100, 231) → (100, 77, 3)
poses_3d = full_pose.reshape(100, 77, 3).contiguous()
print(f"poses_3d shape: {poses_3d.shape}")  # should be (100, 77, 3)

result = soma_layer(
    poses=poses_3d,
    identity_coeffs=identity_coeffs.contiguous(),
    scale_params=scale_params[:, :68].contiguous(),
    transl=transl.contiguous()
)

# Result is a dict — use keys not attributes
mhr_vertices = result['vertices']    # (B, ~18095, 3)
mhr_joints   = result['joints']      # (B, 77, 3)

print(f"✅ MHR vertices: {mhr_vertices.shape}")
print(f"✅ MHR joints:   {mhr_joints.shape}")

# Save
import torch
torch.save({
    'vertices': mhr_vertices,
    'joints':   mhr_joints,
    'transl':   transl
}, '/content/GEM-X/outputs/demo_soma/example1/mhr_params.pt')
print("Saved mhr_params.pt ✓")

EXAMPLE 2

In [ ]:
from soma import SOMALayer
import torch

soma_layer = SOMALayer(identity_model_type='mhr', device='cpu')

# Reshape pose: (90, 231) → (90, 77, 3)
poses_3d = full_pose.reshape(90, 77, 3).contiguous()
print(f"poses_3d shape: {poses_3d.shape}")

result = soma_layer(
    poses=poses_3d,
    identity_coeffs=identity_coeffs.contiguous(),
    scale_params=scale_params[:, :68].contiguous(),
    transl=transl.contiguous()
)

mhr_vertices = result['vertices']
mhr_joints   = result['joints']

print(f"✅ MHR vertices: {mhr_vertices.shape}")
print(f"✅ MHR joints:   {mhr_joints.shape}")

torch.save({
    'vertices': mhr_vertices,
    'joints':   mhr_joints,
    'transl':   transl
}, '/content/GEM-X/outputs/demo_soma/example2/mhr_params.pt')
print("Saved mhr_params.pt ✓")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2, numpy as np, os

out_dir = '/content/GEM-X/outputs/demo_soma/example2/mhr_frames'
os.makedirs(out_dir, exist_ok=True)

verts_all  = mhr_vertices.detach().numpy()   # X=forward, Y=up, Z=lateral
joints_all = mhr_joints.detach().numpy()
n_frames   = verts_all.shape[0]
pad = 0.8

print("Rendering frames...")
for i in range(n_frames):
    verts  = verts_all[i]
    pelvis = joints_all[i, 0]

    px, py, pz = pelvis[0], pelvis[1], pelvis[2]

    fig = plt.figure(figsize=(6, 6), dpi=100)
    ax = fig.add_subplot(111, projection='3d')

    # X=forward → plot as depth (plot-Y), Y=up → plot-Z, Z=lateral → plot-X
    # Negate X so person runs left→right matching input video direction
    ax.scatter(-verts[:, 0], verts[:, 2], verts[:, 1],
               s=0.2, c='steelblue', alpha=0.6)

    ax.set_xlim(-px - pad, -px + pad)   # follow negated X (forward)
    ax.set_ylim(pz - pad, pz + pad)     # lateral
    ax.set_zlim(py - 0.3, py + 1.3)    # height

    ax.view_init(elev=10, azim=270)      # side view: see left-right running
    ax.set_axis_off()
    fig.patch.set_facecolor('white')
    plt.tight_layout()
    plt.savefig(f'{out_dir}/frame_{i:04d}.png', dpi=100, facecolor='white')
    plt.close()
    if i % 10 == 0:
        print(f"  {i}/{n_frames}")

print("Compiling video...")
frame_files = sorted([f for f in os.listdir(out_dir) if f.endswith('.png')])
first = cv2.imread(os.path.join(out_dir, frame_files[0]))
h, w, _ = first.shape

out_path = '/content/GEM-X/outputs/demo_soma/example2/mhr_output.mp4'
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 30, (w, h))
for fname in frame_files:
    writer.write(cv2.imread(os.path.join(out_dir, fname)))
writer.release()
print(f"✅ Saved to {out_path}")